In [3]:
import sys
import os
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Инициализация среды
env = gym.make("CartPole-v1")
n_actions = int(env.action_space.n)
state_dim = env.observation_space.shape[0]

# Создание модели нейронной сети на PyTorch
class DQN(nn.Module):
    def __init__(self, state_dim, n_actions):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, n_actions)
        )

    def forward(self, x):
        return self.net(x)

model = DQN(state_dim, n_actions)

# Функция выбора действия
def get_action(state, epsilon=0):
    if np.random.random() < epsilon:
        return np.random.randint(n_actions)

    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        q_values = model(state_tensor).numpy()[0]
    return int(np.argmax(q_values))

# Проверка
test_tensor = torch.FloatTensor(state_dim).unsqueeze(0)
assert model(test_tensor).shape[1] == n_actions
assert isinstance(model.net[-1], nn.Linear)

s, _ = env.reset()
assert np.shape(get_action(s)) == ()

for eps in [0., 0.1, 0.5, 1.0]:
    actions = [get_action(s, epsilon=eps) for _ in range(10000)]
    state_frequencies = np.bincount(actions, minlength=n_actions)
    best_action = state_frequencies.argmax()
    expected_best = 10000 * (1 - eps + eps / n_actions)
    assert abs(state_frequencies[best_action] - expected_best) < 200
    for other_action in range(n_actions):
        if other_action != best_action:
            expected_other = 10000 * (eps / n_actions)
            assert abs(state_frequencies[other_action] - expected_other) < 200
    print(f'e={eps:.1f} tests passed')

# Компиляция модели
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()
gamma = 0.99

# Функция обучения на батче
def train_on_batch(states, actions, rewards, next_states, dones):
    states = torch.FloatTensor(states)
    next_states = torch.FloatTensor(next_states)
    actions = torch.LongTensor(actions)
    rewards = torch.FloatTensor(rewards)
    dones = torch.FloatTensor(dones)

    current_q = model(states)
    current_q_for_actions = current_q.gather(1, actions.unsqueeze(1)).squeeze(1)

    with torch.no_grad():
        next_q = model(next_states)
        max_next_q = next_q.max(1)[0]
        targets = rewards + (1 - dones) * gamma * max_next_q

    loss = loss_fn(current_q_for_actions, targets)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

# Функция генерации сессии
def generate_session(epsilon=0, train=False, max_steps=500):
    state, _ = env.reset()
    total_reward = 0

    for step in range(max_steps):
        action = get_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        if train:
            train_on_batch(
                np.array([state], dtype=np.float32),
                np.array([action], dtype=np.int64),
                np.array([reward], dtype=np.float32),
                np.array([next_state], dtype=np.float32),
                np.array([float(done)], dtype=np.float32)
            )

        total_reward += reward
        state = next_state

        if done:
            break

    return total_reward

# Основной цикл обучения
epsilon = 0.5

for epoch in range(500):
    epoch_rewards = [generate_session(epsilon=epsilon, train=True) for _ in range(100)]
    mean_reward = np.mean(epoch_rewards)

    epsilon = max(epsilon * 0.99, 0.01)

    print(f"epoch #{epoch:3d}\tmean reward = {mean_reward:.3f}\tepsilon = {epsilon:.3f}")

    if mean_reward > 300:
        print("You Win!")
        break

env.close()

e=0.0 tests passed
e=0.1 tests passed
e=0.5 tests passed
e=1.0 tests passed
epoch #  0	mean reward = 13.200	epsilon = 0.495
epoch #  1	mean reward = 14.290	epsilon = 0.490
epoch #  2	mean reward = 13.950	epsilon = 0.485
epoch #  3	mean reward = 15.690	epsilon = 0.480
epoch #  4	mean reward = 13.750	epsilon = 0.475
epoch #  5	mean reward = 15.200	epsilon = 0.471
epoch #  6	mean reward = 14.950	epsilon = 0.466
epoch #  7	mean reward = 25.780	epsilon = 0.461
epoch #  8	mean reward = 29.230	epsilon = 0.457
epoch #  9	mean reward = 33.810	epsilon = 0.452
epoch # 10	mean reward = 43.440	epsilon = 0.448
epoch # 11	mean reward = 44.300	epsilon = 0.443
epoch # 12	mean reward = 48.330	epsilon = 0.439
epoch # 13	mean reward = 66.430	epsilon = 0.434
epoch # 14	mean reward = 82.250	epsilon = 0.430
epoch # 15	mean reward = 95.480	epsilon = 0.426
epoch # 16	mean reward = 115.510	epsilon = 0.421
epoch # 17	mean reward = 146.450	epsilon = 0.417
epoch # 18	mean reward = 126.620	epsilon = 0.413
epoch # 1